In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit as SS
import tensorflow.python.keras as keras
from matplotlib import pyplot as plt
import os
import tensorflow as tf
import time

In [ ]:
import openpyxl
raw_data = pd.read_excel("36RNA_data.xlsx",engine="openpyxl").to_numpy()
random_index=[i for i in range(2,38)]
np.random.shuffle(random_index)
random_data=raw_data[:,random_index]

In [ ]:
X = (random_data[:1326,:])
print("X：",X)
print(raw_data[:,1])

In [ ]:
# One-Hot Encoding
y_dict = {'O':2,'H':1,'L':0}
Y = np.array([y_dict[i] if i in y_dict else i for i in raw_data[:1326,1]])
y = keras.utils.np_utils.to_categorical(Y)
np.unique(Y,return_counts=True)

In [ ]:
#  Data splitting and model training
from sklearn.metrics import confusion_matrix

flag = 0
train_n = 0

while flag == 0:
    train_n += 1
    start_time = time.time()

    # Build a fully connected network with three softmax neurons
    model = keras.Sequential()
    model.add(keras.layers.Dense(3, use_bias=None, activation="softmax", input_shape=(36,)))
    model.compile(optimizer=keras.optimizers.adam_v2.Adam(lr=0.003),
                  loss='categorical_crossentropy', metrics=['accuracy'])

    # Stratified train-test split
    split = SS(n_splits=1, train_size=0.8)
    for train_index, test_index in split.split(X, y):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

    # Train the model (verbose=0 suppresses training logs)
    history = model.fit(X_train, y_train, batch_size=36, epochs=5000,
                        validation_data=(X_test, y_test), verbose=1)

    y_pred = model.predict(X_test)
    cm = confusion_matrix(
        [list(i).index(max(i)) for i in list(y_test)],
        [list(i).index(max(i)) for i in list(y_pred)]
    )

    acc0 = round(cm[0][0]/(cm[0][0]+cm[0][1]+cm[0][2]), 4)
    acc1 = round(cm[1][1]/(cm[1][0]+cm[1][1]+cm[1][2]), 4)
    acc2 = round(cm[2][2]/(cm[2][0]+cm[2][1]+cm[2][2]), 4)

    print('Run: {}, minority-class correct predictions: {}, class-wise recall: {}/{}/{}, training time (s): {}'
          .format(train_n, cm[1][1], acc0, acc1, acc2, round(time.time()-start_time, 4)))

    # if cm[1][1]>=24:
    if acc1 >= 0.77 and acc2 >= 0.8:
        flag = 1

In [ ]:
#  Model performance evaluation

# Plot accuracy and loss curves
epochs = range(len(history.history['accuracy']))

plt.figure()
plt.plot(epochs, history.history['accuracy'], 'b', label='Training accuracy')
plt.plot(epochs, history.history['val_accuracy'], 'r', label='Validation accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.savefig('Fcnn_acc.png')

plt.figure()
plt.plot(epochs, history.history['loss'], 'b', label='Training loss')
plt.plot(epochs, history.history['val_loss'], 'r', label='Validation loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.savefig('Fcnn_loss.png')

# Plot the confusion matrix
y_pred = model.predict(X_test)
cm = confusion_matrix(
    [list(i).index(max(i)) for i in list(y_test)],
    [list(i).index(max(i)) for i in list(y_pred)]
)
print(cm)
plt.matshow(cm, cmap=plt.cm.Reds)

# Plot ROC curves for each class
from sklearn.metrics import roc_curve, auc

y_test_prob = [list(i).index(max(i)) for i in list(y_test)]

plt.figure(figsize=(5, 5))
for num in range(3):
    y_pred_prob = [i[num] for i in list(y_pred)]
    fpr, tpr, threshold = roc_curve(y_test_prob, y_pred_prob, pos_label=num)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'ROC curve {num} (area = %0.2f)' % roc_auc)

plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Test Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

# Plot predicted probability distribution for class O
list1, list2 = y_test_prob, y_pred_prob
plt.scatter(range(len(list2)), list2, c=list1)
plt.title("Scatter Plot")
plt.xlabel("Index")
plt.ylabel("Values")
cbar = plt.colorbar()
cbar.set_ticks([0, 1, 2])
cbar.set_ticklabels(["L", "H", "O"])
plt.show()

In [ ]:
# Export model predictions and weights

# Export predictions for the training and test sets
header_list = list(pd.read_excel("36RNA_data.xlsx")[:0])
header = [header_list[i] for i in random_index]

for label, X_t, y_t in [['train', X_train, y_train], ['test', X_test, y_test]]:
    data_file = open('Fcnn_pred_{}.txt'.format(label), 'w')
    y_p = model.predict(X_t)

    data_file.writelines('{:12}{:12}{:12}{:12}{:12}{:12}'.format(
        'true_0', 'true_1', 'true_2', 'pred_0', 'pred_1', 'pred_2'))

    for head in header:
        data_file.writelines('{:15}'.format(head))
    data_file.writelines('\n')

    for i in range(len(y_p)):
        data_file.writelines('{:<12}{:<12}{:<12}{:<12.5f}{:<12.5f}{:<12.5f}'.format(
            y_t[i][0], y_t[i][1], y_t[i][2], y_p[i][0], y_p[i][1], y_p[i][2]))

        for j in range(len(X_t[i])):
            data_file.writelines('{:<15.8f}'.format(X_t[i][j]))
        data_file.writelines('\n')

    data_file.close()

# Display model weights
print(model.get_weights())

In [ ]:
# Save the trained model
model.save(filepath='FCN_model.h5')